# Wikipedia Q&A — demo and eval run

`WikiQA` answers questions and checks claims using only Wikipedia, through a Claude tool-use loop:

- **`search_wikipedia(query)`**: ranked results with intros; disambiguation pages are flagged
- **`read_article(title, section?)`**: an article's lead and table of contents, or one section

When a search turns up a disambiguation page, a separate Sonnet 4.6 call decides whether the question could mean more than one entry. If it could, the run stops and returns a clarifying question.

Requires `ANTHROPIC_API_KEY` in the environment or in a `.env` file at the project root.

In [1]:
import json, sys, time
from pathlib import Path

from dotenv import load_dotenv
from IPython.display import HTML, Markdown, display

ROOT = Path.cwd() if (Path.cwd() / "wiki_search").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
load_dotenv(ROOT / ".env")

from wiki_search import WikiQA, wikipedia

## Configuration

`max_articles` sets both the number of results per search and the maximum number of distinct articles opened per question. Raise it for questions that involve many entities.

Wikipedia responses are cached on disk, so reruns are fast and every backbone model sees the same article text.

In [2]:
qa = WikiQA(
    model="claude-sonnet-4-6",
    disambiguation_model="claude-sonnet-4-6",
    max_articles=10,
    max_tool_calls=20,
    effort="medium",
)

# Wikipedia responses are cached in .cache/wikipedia.sqlite so reruns (and model comparisons)
# read identical article text. Use "refresh" to refetch everything, "off" to bypass.
wikipedia.CACHE_MODE = "on"

In [3]:
def tokens(a):
    u = a.usage
    return u["input_tokens"] + u["cache_read_input_tokens"] + u["cache_creation_input_tokens"], u["output_tokens"]


def expected_text(item):
    """One-line summary of an eval item's expected outcome."""
    if item["should_clarify"]:
        return "should ask for clarification" + (f" (or: {item['also_accept']})" if item.get("also_accept") else "")
    if item["category"] == "research":
        return "research brief covering: " + "; ".join(item["must_cover"])
    if item["category"] == "out-of-scope":
        return "should say Wikipedia doesn't cover this"
    parts = [item["expected_verdict"], item["expected_answer"]]
    note = " ⚖️ needs your decision" if item["review"]["status"] == "needs_decision" else ""
    return ": ".join(p for p in parts if p) + note


def render(answer, expected=None, item_id=None):
    """Show one answer: the response (or clarification request), expected answer, and the tool trace."""
    head = f"### {item_id + ': ' if item_id else ''}{answer.question}"
    parts = [head]
    if expected:
        parts.append(f"> **Expected:** {expected}")
    if answer.needs_clarification:
        c = answer.clarification
        opts = "\n".join(f"- {o}" for o in c.options)
        parts.append(f"**🔀 Needs clarification** (disambiguation page *{c.term}*)\n\n{c.question}\n\n{opts}\n\n_Why: {c.rationale}_")
    else:
        parts.append(answer.text)
        parts.extend(f"⚠️ {w}" for w in answer.warnings)
    tin, tout = tokens(answer)
    parts.append(f"<sub>{answer.elapsed_s}s · {answer.usage['llm_calls']} LLM calls · {tin:,} in / {tout:,} out tokens · stop: {answer.stop_reason}</sub>")
    display(Markdown("\n\n".join(parts)))

    steps = []
    for t in answer.trace:
        if t["type"] == "search":
            steps.append(f"🔎 <b>search</b> <code>{t['query']}</code> → {', '.join(t['results'][:5])}{' …' if len(t['results']) > 5 else ''}")
        elif t["type"] == "read":
            steps.append(f"📖 <b>read</b> {t['title']} § {t['section']}")
        elif t["type"] == "disambiguation":
            verdict = "ambiguous → ask user" if t["ambiguous"] else f"resolved → {t['most_likely_title']}"
            steps.append(f"🔀 <b>disambiguation</b> {t['page']}: {verdict}")
        elif t["type"] == "plan":
            steps.append("🗺️ <b>plan research</b>: " + "; ".join(t["subtopics"]))
        elif t["type"] == "revise":
            steps.append(f"✂️ <b>too long</b> ({t['words']} words): asked to shorten")
        elif t["type"] == "submit":
            steps.append("✅ <b>submit_answer</b>" + (f" ({len(t['warnings'])} warnings)" if t["warnings"] else ""))
        elif t["type"] == "error":
            steps.append(f"⚠️ <b>error</b> {t['tool']}: {t['error']}")
    if steps:
        display(HTML("<details><summary>Tool trace (" + str(len(steps)) + " steps)</summary><ol>"
                     + "".join(f"<li>{s}</li>" for s in steps) + "</ol></details><hr>"))

## Single question

In [4]:
answer = qa.ask("Who was the U.S. president when the first person to walk on the Moon was born?")
render(answer)

### Who was the U.S. president when the first person to walk on the Moon was born?

**Answer:** The U.S. president when Neil Armstrong — the first person to walk on the Moon — was born was **Herbert Hoover**. Armstrong was born on August 5, 1930 [1], and Hoover served as the 31st President of the United States from 1929 to 1933 [2].

**Reasoning:**
1. Neil Armstrong, born **August 5, 1930**, was the first person to walk on the Moon as commander of the Apollo 11 mission [1].
2. Herbert Hoover served as the 31st U.S. president from **1929 to 1933**, meaning he was in office on August 5, 1930 [2].

**Sources:**
[1] Neil Armstrong (lead) — https://en.wikipedia.org/wiki/Neil_Armstrong
[2] Herbert Hoover (lead) — https://en.wikipedia.org/wiki/Herbert_Hoover

<sub>12.2s · 3 LLM calls · 10,909 in / 498 out tokens · stop: end_turn</sub>

## Eval set

20 questions: simple facts, claim checks, multi-hop, ambiguous terms, out-of-scope, and two research requests. Each expected answer is backed by a quoted sentence from a named Wikipedia section, pinned to a revision (`evals/verify_evidence.py` re-checks them). Items marked ⚖️ are judgment calls awaiting a decision. Expected outcomes are shown next to each response; automated grading comes later.

In [5]:
eval_set = json.loads((ROOT / "evals" / "eval_set.json").read_text())

# Sequential on purpose: Wikimedia asks clients not to send parallel requests, and bursts get
# throttled (HTTP 429). Cached Wikipedia responses make reruns fast anyway.
wikipedia.stats.clear()
start = time.time()
answers = []
for item in eval_set:
    answers.append(qa.ask(item["question"]))
    print(f"{item['id']}: {answers[-1].elapsed_s}s")
print(f"\nRan {len(eval_set)} questions in {time.time() - start:.0f}s. "
      f"Wikipedia: {wikipedia.stats['requests']} requests, {wikipedia.stats['cache_hits']} cache hits, "
      f"{wikipedia.stats['retries']} throttling retries.")

Wikipedia HTTP 429; retrying in 27s


Wikipedia HTTP 429; retrying in 26s


Wikipedia HTTP 429; retrying in 23s


Wikipedia HTTP 429; retrying in 22s


Wikipedia HTTP 429; retrying in 52s


...

Wikipedia HTTP 429; retrying in 50s


Wikipedia HTTP 429; retrying in 59s


Wikipedia HTTP 429; retrying in 58s


Wikipedia HTTP 429; retrying in 54s


Ran 18 questions in 336s


### Summary

In [6]:
rows = []
for item, a in zip(eval_set, answers):
    tin, tout = tokens(a)
    searches = sum(t["type"] == "search" for t in a.trace)
    reads = sum(t["type"] == "read" for t in a.trace)
    outcome = "clarify" if a.needs_clarification else a.stop_reason
    s = a.structured or {}
    short = " ".join(filter(None, [s.get("verdict"), s.get("answer", "")]))[:60]
    rows.append((item["id"], item["category"], outcome, short, searches, reads, a.elapsed_s, tin, tout))

header = ["id", "category", "outcome", "answer", "searches", "reads", "seconds", "tokens in", "tokens out"]
table = "| " + " | ".join(header) + " |\n|" + "---|" * len(header) + "\n"
table += "\n".join("| " + " | ".join(f"{v:,}" if isinstance(v, int) else str(v) for v in r) + " |" for r in rows)
tot_in, tot_out = sum(r[7] for r in rows), sum(r[8] for r in rows)
# Sonnet 4.6 list price: $3 / $15 per million tokens (cache reads are billed lower, so this is an upper bound).
table += f"\n\n**Total:** {tot_in:,} in / {tot_out:,} out tokens ≈ ${tot_in * 3e-6 + tot_out * 15e-6:.2f} (upper bound)"
display(Markdown(table))

| id | category | outcome | searches | reads | seconds | tokens in | tokens out |
|---|---|---|---|---|---|---|---|
| fact-01 | fact | answered | 1 | 0 | 32.9 | 5,710 | 166 |
| fact-02 | fact | answered | 1 | 0 | 5.6 | 5,344 | 141 |
| fact-03 | fact | answered | 1 | 0 | 4.7 | 5,528 | 141 |
| fact-04 | fact | answered | 1 | 0 | 35.2 | 5,558 | 169 |
| claim-01 | claim | answered | 1 | 1 | 36.2 | 14,923 | 472 |
| claim-02 | claim | answered | 1 | 1 | 34.1 | 9,898 | 469 |
| claim-03 | claim | answered | 1 | 0 | 7.5 | 5,715 | 289 |
| claim-04 | claim | answered | 1 | 1 | 122.4 | 14,401 | 442 |
| claim-05 | claim | answered | 2 | 1 | 118.3 | 14,440 | 572 |
| multi-01 | multi-hop | answered | 4 | 1 | 176.1 | 30,842 | 803 |
| multi-02 | multi-hop | answered | 2 | 1 | 115.4 | 17,165 | 326 |
| multi-03 | multi-hop | answered | 2 | 0 | 117.0 | 7,000 | 272 |
| multi-04 | multi-hop | answered | 2 | 0 | 59.2 | 7,448 | 424 |
| multi-05 | multi-hop | answered | 2 | 0 | 177.0 | 10,876 | 521 |
| ambig-01 | ambiguous | clarify | 1 | 0 | 119.9 | 4,448 | 393 |
| ambig-02 | ambiguous | answered | 1 | 0 | 56.4 | 5,538 | 131 |
| ambig-03 | ambiguous | clarify | 1 | 0 | 62.6 | 4,314 | 399 |
| oos-01 | out-of-scope | answered | 0 | 0 | 3.6 | 1,879 | 110 |

**Total:** 171,027 in / 6,240 out tokens ≈ $0.61 (upper bound)

### Answers

In [7]:
for item, a in zip(eval_set, answers):
    render(a, expected=expected_text(item), item_id=item["id"])

### fact-01: What year did the Eiffel Tower open?

> **Expected:** 1889 (opened for the Exposition Universelle).

**Answer:** The Eiffel Tower opened in **1889** [1]. It was built by Gustave Eiffel's company between 1887 and 1889 and was constructed as the centrepiece of the 1889 World's Fair, commemorating the centennial of the French Revolution [1].

**Sources:**
[1] Eiffel Tower (lead) — https://en.wikipedia.org/wiki/Eiffel_Tower

<sub>32.9s · 2 LLM calls · 5,710 in / 166 out tokens · stop: end_turn</sub>

### fact-02: What is the capital of Australia?

> **Expected:** Canberra.

**Answer:** The capital of Australia is **Canberra**. It is the largest inland city in Australia and the largest population centre in the Australian Capital Territory, with an estimated population of 484,630 as of 2025 [1].

**Sources:**
[1] Canberra (lead) — https://en.wikipedia.org/wiki/Canberra

<sub>5.6s · 2 LLM calls · 5,344 in / 141 out tokens · stop: end_turn</sub>

### fact-03: Who wrote the novel One Hundred Years of Solitude?

> **Expected:** Gabriel García Márquez.

**Answer:** *One Hundred Years of Solitude* was written by Colombian author **Gabriel García Márquez** and published in 1967 [1].

**Sources:**
[1] One Hundred Years of Solitude (lead) — https://en.wikipedia.org/wiki/One_Hundred_Years_of_Solitude

<sub>4.7s · 2 LLM calls · 5,528 in / 141 out tokens · stop: end_turn</sub>

### fact-04: How tall is Mount Kilimanjaro?

> **Expected:** About 5,895 m (19,341 ft) above sea level.

**Answer:** Mount Kilimanjaro stands at **5,895 m (19,341 ft)** above sea level. It is the highest mountain in Africa and the highest free-standing mountain above sea level in the world, rising about 4,900 m (16,100 ft) above its plateau base [1].

**Sources:**
[1] Mount Kilimanjaro (lead) — https://en.wikipedia.org/wiki/Mount_Kilimanjaro

<sub>35.2s · 2 LLM calls · 5,558 in / 169 out tokens · stop: end_turn</sub>

### claim-01: Albert Einstein failed mathematics in school.

> **Expected:** Contradicted: a myth; he excelled at mathematics.

**Answer:** This is a popular myth. According to Wikipedia, Einstein actually *excelled* at mathematics and physics from an early age [1]. He taught himself algebra, calculus, and Euclidean geometry at age twelve and had "mastered integral and differential calculus" by fourteen [1]. When he sat the entrance examination for the ETH polytechnic school at age sixteen, he *did* fail the general part of the test, but he "performed with distinction in physics and mathematics" — the very opposite of failing math [1]. His final school graduation certificate even awarded him the top grade of 6 in algebra, geometry, physics, and history [1].

**Verdict:** Contradicted. Wikipedia states that Einstein excelled at mathematics and physics from a very young age, earned top grades in those subjects, and that the only exam he failed was the *general* portion of an entrance test — not the mathematics portion.

**Sources:**
[1] Albert Einstein § Childhood, youth and education — https://en.wikipedia.org/wiki/Albert_Einstein#Childhood,_youth_and_education

<sub>36.2s · 4 LLM calls · 14,923 in / 472 out tokens · stop: end_turn</sub>

...

### multi-05: Who was the U.S. president when the first person to walk on the Moon was born?

> **Expected:** Neil Armstrong, born 5 August 1930 → Herbert Hoover.

**Answer:** The first person to walk on the Moon was Neil Armstrong, born on August 5, 1930 [1]. At that time, **Herbert Hoover** was the U.S. president, serving his term from 1929 to 1933 as the 31st president of the United States [2].

**Reasoning:**
1. Neil Armstrong (born August 5, 1930) was "the first person to walk on the Moon" as commander of the 1969 Apollo 11 mission [1].
2. Herbert Hoover was the 31st U.S. president, serving from 1929 to 1933, which means he was president on August 5, 1930 — Armstrong's birth date [2].

**Sources:**
[1] Neil Armstrong (lead) — https://en.wikipedia.org/wiki/Neil_Armstrong
[2] Herbert Hoover (lead) — https://en.wikipedia.org/wiki/Herbert_Hoover

<sub>177.0s · 3 LLM calls · 10,876 in / 521 out tokens · stop: end_turn</sub>

### ambig-01: When was Mercury discovered?

> **Expected:** Should ask for clarification (planet vs element; both 'known since antiquity').

**🔀 Needs clarification** (disambiguation page *Mercury*)

Great question! Just to make sure I find the right information for you — are you asking about the **discovery of the planet Mercury** (the closest planet to the Sun) or the **discovery of mercury the element** (the liquid metal, also known as quicksilver)?

- Mercury (planet)
- Mercury (element)

_Why: The question "When was Mercury discovered?" could reasonably refer to either the planet Mercury or the element mercury. Both have historical "discovery" contexts that would lead to materially different answers. The planet Mercury has been known since antiquity but was studied in detail over centuries, while the element mercury (quicksilver) also has ancient origins but a different history. Both interpretations are plausible without further context._

<sub>119.9s · 2 LLM calls · 4,448 in / 393 out tokens · stop: needs_clarification</sub>

### ambig-02: How many studio albums did Queen release?

> **Expected:** Should NOT ask: Queen the band → 15 studio albums.

**Answer:** The British rock band Queen released **15 studio albums** [1].

**Sources:**
[1] Queen discography (lead) — https://en.wikipedia.org/wiki/Queen_discography

<sub>56.4s · 2 LLM calls · 5,538 in / 131 out tokens · stop: end_turn</sub>

### ambig-03: What is the population of Springfield?

> **Expected:** Should ask for clarification (many Springfields).

**🔀 Needs clarification** (disambiguation page *Springfield*)

There are many places named Springfield! Could you clarify which one you mean? For example, are you asking about Springfield, Missouri (the most populous city named Springfield), Springfield, Illinois (the state capital), Springfield, Massachusetts (the first Springfield settled in the US), or another Springfield entirely?

- Springfield, Missouri
- Springfield, Illinois
- Springfield, Massachusetts
- Springfield, Ohio
- Springfield, Oregon
- Springfield, Virginia

_Why: The question asks for the population of "Springfield" without any additional context. There are dozens of real cities and towns named Springfield across the US and the world, several of which are notable enough to have distinct Wikipedia articles with population data. The most likely candidates include Springfield, MO (most populous), Springfield, IL (state capital), Springfield, MA (historically significant), and Springfield, OH, but without more context it's genuinely unclear which one the user means._

<sub>62.6s · 2 LLM calls · 4,314 in / 399 out tokens · stop: needs_clarification</sub>

### oos-01: What will the weather be in Paris tomorrow?

> **Expected:** Should say Wikipedia doesn't cover forecasts; no fabricated answer.

I'm sorry, but I can't answer questions about weather forecasts. My only source of information is English Wikipedia, which does not contain real-time or forecast weather data. For an accurate weather forecast for Paris tomorrow, I'd recommend checking a dedicated weather service such as:

- **weather.com**
- **météo-france.fr** (France's national meteorological service)
- **forecast.io / Dark Sky**
- Or simply searching "Paris weather tomorrow" in your preferred search engine.

<sub>3.6s · 1 LLM calls · 1,879 in / 110 out tokens · stop: end_turn</sub>

## Research brief

For overviews, histories and comparisons, the agent calls `plan_research`, which raises its budget (`research_max_articles`, `research_max_tool_calls`), reads several articles in parallel, and submits a brief: overview, sections, optional comparison table, how items were chosen, and further reading. Briefs over ~600 words are sent back once to shorten.

In [ ]:
brief = qa.ask("Compare the 5 most common statistical distributions and summarize what they are best suited to model")
print("mode:", brief.mode)
render(brief)

## Answering after a clarification

When the agent asks for clarification, call `ask` again with the user's reply. The CLI does this interactively.

In [8]:
first = qa.ask("When was Mercury discovered?")
if first.needs_clarification:
    print("Agent asked:", first.clarification.question)
    followup = qa.ask("When was Mercury discovered?", clarification='By "Mercury" I mean the planet.')
    render(followup)
else:
    render(first)

Agent asked: Great question! Just to make sure I find the right information, could you clarify which "Mercury" you're asking about? Are you referring to **Mercury the planet** (the closest planet to the Sun) or **Mercury the element** (the chemical element, also known as quicksilver)?


### When was Mercury discovered?

(Clarification from the user: By "Mercury" I mean the planet.)

**Answer:** Mercury was not "discovered" at a single point in time — it is visible to the naked eye and has been known since antiquity. The earliest known recorded observations of Mercury come from Assyrian astronomers around the **14th century BC**, recorded on the MUL.APIN tablets. Babylonian records of the planet also date back to the 1st millennium BC [1]. The first **telescopic** observations of Mercury were made by Thomas Harriot and Galileo in **1610** [1].

**Assumptions & gaps:** Because Mercury is visible to the naked eye, there is no single "discovery" moment as there is for planets like Uranus or Neptune. Ancient civilizations across Mesopotamia, China, India, and the Maya world all independently recognized it as a distinct celestial body long before the telescope era.

**Sources:**
[1] Mercury (planet) § Observation history — https://en.wikipedia.org/wiki/Mercury_(planet)#Observation_history

<sub>10.8s · 3 LLM calls · 11,202 in / 377 out tokens · stop: end_turn</sub>